# Middle Mile Analysis

Notebook ini melakukan analisis biaya T&W dan ODS serta mengevaluasi rute middle mile berdasarkan koordinat.

In [6]:
import pandas as pd
import numpy as np
import re

# Fungsi bantu
def clean_currency(value):
    if pd.isnull(value):
        return np.nan
    try:
        clean = re.sub(r"[^\d]", "", str(value))
        if len(clean) > 9:
            clean = clean[-9:]
        return float(clean) / 1_000_000
    except:
        return np.nan

def normalize_coord(val):
    try:
        val = float(val)
        return val / 1_000_000 if abs(val) > 1000 else val
    except:
        return np.nan


## 1. Load Dataset

In [7]:
# Ganti path jika perlu
file_path = "1742574481558_Dataset-ME-2025.xlsx"

tw_cost = pd.read_excel(file_path, sheet_name="T&W Cost")
ods_cost = pd.read_excel(file_path, sheet_name="ODS Cost")
routes = pd.read_excel(file_path, sheet_name="Middle Mile")

# Bersihkan kolom tak bernama
tw_cost = tw_cost.loc[:, ~tw_cost.columns.str.contains("Unnamed", na=False)]
ods_cost = ods_cost.loc[:, ~ods_cost.columns.str.contains("Unnamed", na=False)]
routes = routes.loc[:, ~routes.columns.str.contains("Unnamed", na=False)]

tw_cost.head()


,Distributor Area,Area,Type,Vehicle,Rate,Trips/mth,Total Cost
0,Bali Nusra,Denpasar,Direct CPU,Build Up,12150000.0,10,121500000.0
1,Bali Nusra,Denpasar,Direct CPU,FUSO,8700000.0,4,34800000.0
2,Bali Nusra,Lombok,Direct CPU,Build Up,21440000.0,10,214400000.0
3,Bali Nusra,Lombok,Direct CPU,FUSO,16240000.0,5,81200000.0
4,Bali Nusra,Lombok,Direct CPU,CDD,12100000.0,2,24200000.0


## 2. Preprocessing & Normalisasi

In [8]:
# Normalisasi biaya
tw_cost["TnW_Cost_Million"] = tw_cost["Total Cost"].apply(clean_currency)
ods_cost["ODS_Cost_Million"] = ods_cost["TOTAL COST"].apply(clean_currency)

# Normalisasi koordinat
coord_cols = ['Origin Latitude', 'Longitude_x', 'Destination Latitude', 'Destination Longitude']
for col in coord_cols:
    routes[col] = routes[col].apply(normalize_coord)

routes.head()


,Distributor Area,Origin Area,Origin Latitude,Longitude_x,Destination Area,Destination Latitude,Destination Longitude,Volume (CS)
0,Kalimantan,Barabai,-2.581936,115.390569,Ampah - Teweh,-0.935414,114.901123,50
1,Kalimantan,Batulicin,-3.455801,11.599749,Ampah - Teweh,-0.935414,114.901123,50
2,Kalimantan,Bontang-Sangata,0.505968,117.532558,Ampah - Teweh,-0.935414,114.901123,50
3,Kalimantan,Ketapang,-0.015698,1.105215,Ampah - Teweh,-0.935414,114.901123,50
4,Kalimantan,Pangkalanbun,-2.684718,111.631067,Ampah - Teweh,-0.935414,114.901123,50


## 3. Validasi Koordinat dan Hitung Jarak

In [9]:
from geopy.distance import geodesic

# Validasi koordinat
def is_valid_coord(lat, lon):
    return -90 <= lat <= 90 and -180 <= lon <= 180

valid_routes = []
distances = []

for _, row in routes.iterrows():
    o_lat, o_lon = row['Origin Latitude'], row['Longitude_x']
    d_lat, d_lon = row['Destination Latitude'], row['Destination Longitude']

    if all(map(is_valid_coord, [o_lat, d_lat], [o_lon, d_lon])):
        try:
            dist = geodesic((o_lat, o_lon), (d_lat, d_lon)).km
            valid_routes.append({**row, "Distance_km": dist})
            distances.append(dist)
        except:
            continue

valid_df = pd.DataFrame(valid_routes)
valid_df.head()


,Distributor Area,Origin Area,Origin Latitude,Longitude_x,Destination Area,Destination Latitude,Destination Longitude,Volume (CS),Distance_km
0,Kalimantan,Barabai,-2.581936,115.390569,Ampah - Teweh,-0.935414,114.901123,50,190.034792
1,Kalimantan,Batulicin,-3.455801,11.599749,Ampah - Teweh,-0.935414,114.901123,50,11490.071762
2,Kalimantan,Bontang-Sangata,0.505968,117.532558,Ampah - Teweh,-0.935414,114.901123,50,333.472997
3,Kalimantan,Ketapang,-0.015698,1.105215,Ampah - Teweh,-0.935414,114.901123,50,12667.293676
4,Kalimantan,Pangkalanbun,-2.684718,111.631067,Ampah - Teweh,-0.935414,114.901123,50,412.049596


## 4. Summary Statistik

In [ ]:
print("📌 Jumlah rute dengan koordinat valid:", len(valid_df))
print("📌 Rata-rata jarak antar rute: {:.2f} km".format(np.mean(distances) if distances else float("nan")))

print("\n💰 Ringkasan Biaya T&W:")
print(tw_cost.groupby("Distributor Area")["TnW_Cost_Million"].sum())

print("\n🚛 Ringkasan Biaya ODS:")
print(ods_cost.groupby("Distributor Area")["ODS_Cost_Million"].sum())


📌 Jumlah rute dengan koordinat valid: 994
📌 Rata-rata jarak antar rute: 2100.30 km

💰 Ringkasan Biaya T&W:
Distributor Area
Bali Nusra    1761.000000
Kalimantan    7977.894519
Sulawesi      2834.900000
Name: TnW_Cost_Million, dtype: float64

🚛 Ringkasan Biaya ODS:
Distributor Area
Bali Nusra    3712.824553
Kalimantan    4463.281591
Sulawesi      5645.950313
Name: ODS_Cost_Million, dtype: float64


In [16]:

# Load Middle Mile, T&W Cost, and ODS Cost sheets
middle_mile = pd.read_excel(file_path, sheet_name="Middle Mile")
tw_cost = pd.read_excel(file_path, sheet_name="T&W Cost")
ods_cost = pd.read_excel(file_path, sheet_name="ODS Cost")

# Clean up T&W cost data
tw_cost_cleaned = tw_cost[['Distributor Area', 'Area', 'Vehicle', 'Rate', 'Trips/mth']].copy()
tw_cost_cleaned = tw_cost_cleaned.dropna(subset=['Rate', 'Trips/mth'])

# Calculate cost per trip and average stats
tw_cost_cleaned['Rate'] = pd.to_numeric(tw_cost_cleaned['Rate'], errors='coerce')
tw_cost_cleaned['Trips/mth'] = pd.to_numeric(tw_cost_cleaned['Trips/mth'], errors='coerce')
tw_cost_cleaned.dropna(inplace=True)
tw_cost_cleaned['Monthly Cost'] = tw_cost_cleaned['Rate'] * tw_cost_cleaned['Trips/mth']

# Summary per vehicle type
vehicle_summary = tw_cost_cleaned.groupby('Vehicle').agg(
    Avg_Rate=('Rate', 'mean'),
    Avg_Trips_per_Month=('Trips/mth', 'mean'),
    Total_Monthly_Cost=('Monthly Cost', 'sum'),
    Count=('Vehicle', 'count')
).sort_values(by='Total_Monthly_Cost', ascending=False)

# Extract common destination summary from middle mile
destination_counts = middle_mile['Destination Area'].value_counts().reset_index()
destination_counts.columns = ['Destination Area', 'Route Count']